In [ ]:
# NotY3dGenAI - Professional AI-Powered 3D Model Generator
# Complete working version with proper 3D generation

import os
import sys
import time
import torch
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import plotly.graph_objects as go
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
from google.colab import drive, files
import json
import base64
import warnings
warnings.filterwarnings('ignore')

# Mount Google Drive
drive.mount('/content/drive')

print("📦 Installing required packages...")
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers accelerate diffusers
!pip install -q opencv-python-headless
!pip install -q trimesh
!pip install -q open3d
!pip install -q scikit-learn
!pip install -q rembg
!pip install -q xatlas

# Import required libraries
import cv2
from scipy import ndimage
from pathlib import Path
import trimesh
import open3d as o3d
from sklearn.neighbors import LocalOutlierFactor

print("✅ All packages installed!")

# Create directories
Path("/content/noty3d_models").mkdir(exist_ok=True)
Path("/content/drive/MyDrive/NotY3D_Models").mkdir(exist_ok=True)

class True3DGenerator:
    """Professional AI-powered 3D model generator"""
    
    def __init__(self):
        self.device = self.setup_device()
        self.setup_models()
        
    def setup_device(self):
        if torch.cuda.is_available():
            device = torch.device("cuda")
            print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
            torch.cuda.empty_cache()
            return device
        return torch.device("cpu")
    
    def setup_models(self):
        """Setup Stable Diffusion for image generation"""
        print("🔄 Loading AI models...")
        
        try:
            from diffusers import StableDiffusionPipeline
            
            # Main pipeline for image generation
            self.pipeline = StableDiffusionPipeline.from_pretrained(
                "runwayml/stable-diffusion-v1-5",
                torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
                safety_checker=None,
                requires_safety_checker=False
            )
            
            if torch.cuda.is_available():
                self.pipeline = self.pipeline.to("cuda")
                self.pipeline.enable_attention_slicing()
            
            print("✅ AI models loaded successfully!")
            self.models_loaded = True
            
        except Exception as e:
            print(f"⚠️ Model loading warning: {e}")
            print("Using enhanced procedural generation...")
            self.pipeline = None
            self.models_loaded = False
    
    def generate_multiview_images(self, prompt, num_views=8):
        """Generate multi-view images"""
        images = []
        angles = np.linspace(0, 360, num_views, endpoint=False)
        
        print(f"📸 Generating {num_views} views...")
        
        for i, angle in enumerate(angles):
            print(f"   View {i+1}/{num_views} - Angle: {angle:.0f}°")
            
            # Add camera angle to prompt
            angle_prompt = f"professional 3D render of {prompt}, camera angle {angle:.0f} degrees, high quality, detailed, realistic, 4K, cinematic lighting"
            
            if self.models_loaded and self.pipeline:
                with torch.autocast("cuda" if torch.cuda.is_available() else "cpu"):
                    result = self.pipeline(
                        angle_prompt,
                        num_inference_steps=35,
                        guidance_scale=7.5,
                        height=512,
                        width=512
                    )
                    image = result.images[0]
            else:
                # Create enhanced procedural image
                image = self.create_procedural_3d_image(prompt, angle)
            
            images.append(image)
        
        return images
    
    def create_procedural_3d_image(self, prompt, angle):
        """Create high-quality procedural 3D-like image"""
        img = Image.new('RGB', (512, 512), color=(30, 30, 50))
        draw = ImageDraw.Draw(img)
        
        center_x, center_y = 256, 256
        prompt_lower = prompt.lower()
        
        # Create 3D-looking shapes based on prompt
        if 'man' in prompt_lower or 'human' in prompt_lower or 'person' in prompt_lower:
            # Draw 3D human figure with shading
            # Head (sphere-like)
            head_size = 45
            for offset in range(3):
                shade = 200 - offset * 20
                draw.ellipse(
                    [center_x - head_size + offset, center_y - 85 - offset,
                     center_x + head_size - offset, center_y - 15 + offset],
                    fill=(shade, shade-20, shade-40),
                    outline=(100, 80, 60)
                )
            
            # Body with 3D effect
            body_width = 55
            for offset in range(3):
                shade = 180 - offset * 20
                draw.rectangle(
                    [center_x - body_width + offset, center_y - 10 + offset,
                     center_x + body_width - offset, center_y + 120 - offset],
                    fill=(shade, shade-30, shade-50),
                    outline=(100, 80, 60)
                )
            
            # Arms with 3D shading
            arm_width = 25
            for offset in range(3):
                shade = 170 - offset * 20
                # Left arm
                draw.rectangle(
                    [center_x - 85 + offset, center_y + 10 + offset,
                     center_x - 55 - offset, center_y + 80 - offset],
                    fill=(shade, shade-30, shade-50),
                    outline=(100, 80, 60)
                )
                # Right arm
                draw.rectangle(
                    [center_x + 55 + offset, center_y + 10 + offset,
                     center_x + 85 - offset, center_y + 80 - offset],
                    fill=(shade, shade-30, shade-50),
                    outline=(100, 80, 60)
                )
            
            # Add authoritative details (coat/tie)
            draw.rectangle(
                [center_x - 20, center_y + 20, center_x + 20, center_y + 60],
                fill=(40, 40, 60), outline=(30, 30, 50)
            )
            
        elif 'dragon' in prompt_lower or 'creature' in prompt_lower:
            # Dragon figure
            # Body
            draw.ellipse([center_x-80, center_y-40, center_x+80, center_y+80],
                        fill=(80, 40, 30), outline=(60, 30, 20))
            # Head
            draw.ellipse([center_x+40, center_y-60, center_x+100, center_y-10],
                        fill=(100, 50, 40), outline=(80, 40, 30))
            # Wings
            draw.polygon([(center_x-40, center_y-20), (center_x-120, center_y-100),
                         (center_x-60, center_y-40)], fill=(70, 35, 25))
            draw.polygon([(center_x+40, center_y-20), (center_x+120, center_y-100),
                         (center_x+60, center_y-40)], fill=(70, 35, 25))
        
        # Add angle indicator for 3D rotation feel
        angle_rad = np.radians(angle)
        radius = 120
        end_x = center_x + radius * np.cos(angle_rad)
        end_y = center_y + radius * np.sin(angle_rad)
        
        # Draw rotation arc
        for a in range(0, 360, 30):
            a_rad = np.radians(a)
            x = center_x + radius * np.cos(a_rad)
            y = center_y + radius * np.sin(a_rad)
            draw.ellipse([x-3, y-3, x+3, y+3], fill=(100, 100, 150))
        
        draw.line([(center_x, center_y), (end_x, end_y)], fill=(255, 100, 100), width=4)
        
        # Add lighting effect
        for i in range(50):
            x = np.random.randint(0, 512)
            y = np.random.randint(0, 512)
            intensity = np.random.randint(100, 255)
            draw.point((x, y), fill=(intensity, intensity, intensity))
        
        return img
    
    def estimate_depth_map(self, image):
        """Generate high-quality depth map"""
        img_array = np.array(image)
        gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
        
        # Multi-scale depth estimation
        depths = []
        for scale in [1, 2, 3]:
            scaled = cv2.resize(gray, (gray.shape[1]//scale, gray.shape[0]//scale))
            
            # Sobel gradients
            sobelx = cv2.Sobel(scaled, cv2.CV_64F, 1, 0, ksize=5)
            sobely = cv2.Sobel(scaled, cv2.CV_64F, 0, 1, ksize=5)
            depth = np.sqrt(sobelx**2 + sobely**2)
            
            # Resize back
            depth = cv2.resize(depth, (gray.shape[1], gray.shape[0]))
            depths.append(depth)
        
        # Combine scales
        depth = np.mean(depths, axis=0)
        
        # Apply edge preservation
        edges = cv2.Canny(gray, 30, 100)
        depth = depth + edges.astype(np.float64) * 0.2
        
        # Normalize
        depth = (depth - depth.min()) / (depth.max() - depth.min() + 1e-6)
        
        # Smooth
        depth = cv2.bilateralFilter(depth.astype(np.float32), 11, 50, 50)
        depth = cv2.GaussianBlur(depth, (7, 7), 0)
        
        return depth
    
    def depth_to_pointcloud_advanced(self, depth, image, angle):
        """Convert depth map to point cloud with better quality"""
        h, w = depth.shape
        focal_length = w / (2 * np.tan(np.radians(60) / 2))
        
        # Create coordinate grid
        x, y = np.meshgrid(np.arange(w), np.arange(h))
        
        # Calculate 3D coordinates
        X = (x - w/2) * depth / focal_length
        Y = (y - h/2) * depth / focal_length
        Z = depth * 3  # Enhanced depth scaling
        
        points = np.stack([X.flatten(), Y.flatten(), Z.flatten()], axis=1)
        
        # Get vertex colors
        img_array = np.array(image)
        colors = img_array.reshape(-1, 3) / 255.0
        
        # Intelligent sampling based on depth variation
        depth_variation = np.std(depth)
        if depth_variation > 0.1:
            step = 2  # Keep more points for detailed areas
        else:
            step = 3  # Sample more aggressively for flat areas
        
        points = points[::step]
        colors = colors[::step]
        
        # Apply camera rotation
        theta = np.radians(angle)
        rot_y = np.array([
            [np.cos(theta), 0, np.sin(theta)],
            [0, 1, 0],
            [-np.sin(theta), 0, np.cos(theta)]
        ])
        
        points = points @ rot_y.T
        
        return points, colors
    
    def create_mesh_from_pointcloud(self, points, colors, detail_level=1.0):
        """Create high-quality mesh from point cloud"""
        
        print(f"   Processing {len(points)} points...")
        
        # Remove statistical outliers
        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(points)
        pcd.colors = o3d.utility.Vector3dVector(colors)
        
        # Statistical outlier removal
        pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=30, std_ratio=2.0)
        
        # Voxel downsampling for even distribution
        voxel_size = 0.03 if detail_level > 0.8 else 0.05
        pcd = pcd.voxel_down_sample(voxel_size)
        
        print(f"   After cleanup: {len(pcd.points)} points")
        
        # Estimate normals
        pcd.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.15, max_nn=40))
        pcd.orient_normals_consistent_tangent_plane(k=20)
        
        # Poisson surface reconstruction
        depth_val = min(9, int(7 * detail_level))
        mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
            pcd, depth=depth_val, width=0, scale=1.1, linear_fit=True
        )
        
        # Filter low-density vertices
        densities_array = np.asarray(densities)
        density_threshold = np.percentile(densities_array, 15)
        vertices_to_remove = densities_array < density_threshold
        mesh.remove_vertices_by_mask(vertices_to_remove)
        
        # Remove degenerate faces
        mesh.remove_degenerate_triangles()
        mesh.remove_unreferenced_vertices()
        
        # Smooth mesh
        mesh = mesh.filter_smooth_taubin(number_of_iterations=10)
        
        # Compute vertex normals for proper shading
        mesh.compute_vertex_normals()
        
        return mesh

class NotY3dGenAI:
    """Main application"""
    
    def __init__(self):
        self.generator = True3DGenerator()
        self.current_mesh = None
        self.current_model_path = None
        self.create_ui()
    
    def create_ui(self):
        """Create professional UI"""
        
        display(HTML("""
        <style>
            @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap');
            
            * { font-family: 'Inter', sans-serif; }
            
            .title {
                font-size: 52px;
                font-weight: 800;
                background: linear-gradient(135deg, #667eea 0%, #764ba2 50%, #f093fb 100%);
                -webkit-background-clip: text;
                -webkit-text-fill-color: transparent;
                text-align: center;
                margin-bottom: 10px;
            }
            
            .subtitle {
                text-align: center;
                color: #888;
                margin-bottom: 30px;
            }
            
            .info {
                background: linear-gradient(135deg, #667eea, #764ba2);
                color: white;
                padding: 20px;
                border-radius: 15px;
                margin: 20px 0;
                box-shadow: 0 10px 30px rgba(0,0,0,0.2);
            }
            
            .btn-generate {
                background: linear-gradient(135deg, #667eea, #764ba2);
                color: white;
                border: none;
                padding: 15px;
                font-size: 18px;
                font-weight: 600;
                border-radius: 10px;
                cursor: pointer;
                width: 100%;
                transition: transform 0.2s;
            }
            
            .btn-generate:hover { transform: translateY(-2px); }
            
            .btn-download {
                background: linear-gradient(135deg, #11998e, #38ef7d);
                color: white;
                border: none;
                padding: 12px;
                font-size: 16px;
                font-weight: 600;
                border-radius: 8px;
                cursor: pointer;
                width: 100%;
                margin-top: 10px;
            }
            
            .btn-download:hover { transform: translateY(-2px); }
            
            .status {
                background: #2d2d2d;
                color: #0f0;
                padding: 10px;
                border-radius: 5px;
                font-family: monospace;
                margin-top: 10px;
                white-space: pre-wrap;
            }
            
            .widget-label {
                color: white !important;
            }
        </style>
        """))
        
        display(HTML('<div class="title">🎨 NotY3dGenAI</div>'))
        display(HTML('<div class="subtitle">Professional AI-Powered 3D Model Generator</div>'))
        
        display(HTML(f"""
        <div class="info">
            <div style="display: flex; justify-content: space-between;">
                <div>
                    ✨ <b>Device:</b> {self.generator.device}<br>
                    🎯 <b>Model:</b> Stable Diffusion + 3D Reconstruction
                </div>
                <div>
                    ⚡ <b>Status:</b> Ready<br>
                    💾 <b>Storage:</b> Google Drive
                </div>
            </div>
        </div>
        """))
        
        # Controls
        self.prompt = widgets.Textarea(
            value="A powerful authoritative man with muscular build, confident posture, wearing dark formal attire, serious expression, cinematic lighting, high quality 3D render",
            placeholder="Describe your 3D model in detail...",
            layout=widgets.Layout(width='100%', height='120px')
        )
        
        self.quality = widgets.Dropdown(
            options=['Ultra', 'High', 'Medium', 'Low'],
            value='High',
            description='Quality:',
            style={'description_width': 'initial'}
        )
        
        self.polygons = widgets.IntSlider(
            value=30000,
            min=10000,
            max=80000,
            step=5000,
            description='Max Polygons:',
            style={'description_width': 'initial'}
        )
        
        self.generate_btn = widgets.Button(
            description="🚀 GENERATE 3D MODEL",
            layout=widgets.Layout(width='100%', height='50px')
        )
        
        self.download_btn = widgets.Button(
            description="📥 DOWNLOAD MODEL",
            layout=widgets.Layout(width='100%', height='40px'),
            disabled=True
        )
        
        self.status = widgets.Output()
        
        controls = widgets.VBox([
            self.prompt,
            self.quality,
            self.polygons,
            self.generate_btn,
            self.download_btn,
            self.status
        ], layout=widgets.Layout(padding='10px'))
        
        self.viewer = widgets.Output()
        
        container = widgets.HBox([
            widgets.VBox([controls], layout=widgets.Layout(width='40%')),
            widgets.VBox([
                widgets.HTML("<h3 style='text-align:center; color:white;'>🎬 3D Model Viewer</h3>"),
                self.viewer
            ], layout=widgets.Layout(width='60%'))
        ])
        
        display(container)
        
        self.generate_btn.on_click(self.generate)
        self.download_btn.on_click(self.download)
        
        with self.viewer:
            self.show_placeholder()
    
    def show_placeholder(self):
        fig = go.Figure()
        fig.add_annotation(
            text="✨ Your 3D model will appear here ✨<br><br>Click GENERATE to start",
            x=0.5, y=0.5, showarrow=False,
            font=dict(size=16, color="gray")
        )
        fig.update_layout(
            height=500,
            paper_bgcolor='black',
            plot_bgcolor='black',
            margin=dict(l=0, r=0, b=0, t=0)
        )
        fig.show()
    
    def generate(self, btn):
        with self.status:
            clear_output()
            start_time = time.time()
            
            print("🎨 Starting professional 3D model generation...")
            print(f"📝 Prompt: {self.prompt.value[:100]}...")
            print(f"⚙️ Quality: {self.quality.value}")
            print("-" * 50)
            
            try:
                # Quality settings
                quality_map = {'Ultra': 1.2, 'High': 1.0, 'Medium': 0.7, 'Low': 0.5}
                detail = quality_map[self.quality.value]
                num_views = 8 if self.quality.value in ['Ultra', 'High'] else 6
                
                # Step 1: Generate multi-view images
                print("\n📸 STEP 1/4: Generating multi-view images...")
                images = self.generator.generate_multiview_images(self.prompt.value, num_views)
                
                # Display a sample
                display(HTML("<b>📸 Sample generated view:</b>"))
                display(images[0].resize((256, 256)))
                
                # Step 2: Generate depth maps and point clouds
                print("\n🗺️ STEP 2/4: Creating depth maps and point clouds...")
                all_points = []
                all_colors = []
                angles = np.linspace(0, 360, num_views, endpoint=False)
                
                for i, (img, angle) in enumerate(zip(images, angles)):
                    print(f"   Processing view {i+1}/{num_views}...")
                    depth = self.generator.estimate_depth_map(img)
                    points, colors = self.generator.depth_to_pointcloud_advanced(depth, img, angle)
                    all_points.append(points)
                    all_colors.append(colors)
                
                # Combine all point clouds
                combined_points = np.vstack(all_points)
                combined_colors = np.vstack(all_colors)
                
                # Step 3: Create mesh
                print("\n🔺 STEP 3/4: Reconstructing 3D mesh...")
                mesh = self.generator.create_mesh_from_pointcloud(combined_points, combined_colors, detail)
                
                # Simplify if needed
                if len(mesh.triangles) > self.polygons.value:
                    print(f"   Simplifying mesh: {len(mesh.triangles)} -> {self.polygons.value} faces")
                    mesh = mesh.simplify_quadric_decimation(self.polygons.value)
                
                # Save model
                print("\n💾 Saving model...")
                timestamp = int(time.time())
                safe_prompt = "".join(c for c in self.prompt.value[:30] if c.isalnum() or c == ' ')
                filename = f"noty3d_{safe_prompt}_{timestamp}.obj"
                local_path = f"/content/noty3d_models/{filename}"
                
                # Export mesh
                o3d.io.write_triangle_mesh(local_path, mesh)
                
                # Save to Drive
                drive_path = f"/content/drive/MyDrive/NotY3D_Models/{filename}"
                import shutil
                shutil.copy(local_path, drive_path)
                
                self.current_model_path = local_path
                self.current_mesh = mesh
                
                # Calculate time
                elapsed = time.time() - start_time
                
                print("\n" + "=" * 50)
                print("✅ MODEL GENERATED SUCCESSFULLY!")
                print("=" * 50)
                print(f"⏱️ Time: {elapsed:.1f} seconds")
                print(f"🔺 Vertices: {len(mesh.vertices):,}")
                print(f"🔻 Faces: {len(mesh.triangles):,}")
                print(f"💾 Saved: {local_path}")
                print(f"☁️ Drive backup: {drive_path}")
                
                # Enable download button
                self.download_btn.disabled = False
                
                # Display in 3D viewer
                with self.viewer:
                    clear_output()
                    self.display_3d_professional(mesh)
                
                print("\n🎉 Generation complete! Click the DOWNLOAD button to save your model.")
                
            except Exception as e:
                print(f"\n❌ Error: {str(e)}")
                import traceback
                traceback.print_exc()
                print("\n💡 Tip: Try reducing quality or using a simpler prompt")
    
    def display_3d_professional(self, mesh):
        """Professional 3D viewer with WebGL"""
        try:
            vertices = np.asarray(mesh.vertices)
            triangles = np.asarray(mesh.triangles)
            
            # Sample for performance if needed
            if len(vertices) > 15000:
                step = len(vertices) // 10000
                vertices = vertices[::step]
                triangles = triangles[::max(1, step//2)]
            
            # Create advanced 3D visualization with proper shading
            fig = go.Figure(data=[
                go.Mesh3d(
                    x=vertices[:, 0],
                    y=vertices[:, 1],
                    z=vertices[:, 2],
                    i=triangles[:, 0],
                    j=triangles[:, 1],
                    k=triangles[:, 2],
                    intensity=vertices[:, 2],
                    colorscale='Viridis',
                    opacity=0.95,
                    lighting=dict(
                        ambient=0.6,
                        diffuse=0.8,
                        specular=0.7,
                        roughness=0.3,
                        fresnel=0.2
                    ),
                    lightposition=dict(x=200, y=200, z=400),
                    flatshading=False
                )
            ])
            
            # Professional camera setup
            fig.update_layout(
                scene=dict(
                    camera=dict(
                        eye=dict(x=1.8, y=1.8, z=1.8),
                        up=dict(x=0, y=1, z=0),
                        center=dict(x=0, y=0, z=0)
                    ),
                    aspectmode='data',
                    bgcolor='#0a0a0a',
                    xaxis=dict(visible=False, showgrid=False, showbackground=False),
                    yaxis=dict(visible=False, showgrid=False, showbackground=False),
                    zaxis=dict(visible=False, showgrid=False, showbackground=False)
                ),
                width=750,
                height=550,
                margin=dict(l=0, r=0, b=0, t=0),
                paper_bgcolor='#0a0a0a',
                plot_bgcolor='#0a0a0a',
                title={
                    'text': "🎨 3D Model Preview - Drag to rotate | Scroll to zoom",
                    'x': 0.5,
                    'y': 0.98,
                    'font': {'size': 14, 'color': 'white'}
                }
            )
            
            fig.show()
            
        except Exception as e:
            print(f"⚠️ Viewer error: {e}")
            print("Model saved successfully, but preview unavailable")
    
    def download(self, btn):
        """Download the generated model"""
        if hasattr(self, 'current_model_path') and self.current_model_path and os.path.exists(self.current_model_path):
            print("📥 Preparing download...")
            files.download(self.current_model_path)
            print("✅ Download started! Check your browser's download folder.")
        else:
            print("❌ No model available. Please generate a model first.")

# Launch the application
print("""
╔══════════════════════════════════════════════════════════════════╗
║                                                                  ║
║              🎨 NotY3dGenAI - Professional 3D AI                ║
║                                                                  ║
║  ⚡ Features:                                                    ║
║  • True AI-powered 3D model generation                          ║
║  • Multi-view reconstruction                                     ║
║  • Professional WebGL 3D viewer                                 ║
║  • High-quality mesh generation                                 ║
║  • Full quality & polygon control                               ║
║  • One-click download                                           ║
║  • Auto-save to Google Drive                                    ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
""")

# Create and launch app
app = NotY3dGenAI()

print("\n" + "=" * 70)
print("✨ NotY3dGenAI is READY for professional 3D generation!")
print("=" * 70)
print("\n💡 Tips for best results:")
print("   • Be very descriptive in your prompts")
print("   • Use 'Ultra' or 'High' quality for detailed models")
print("   • Higher polygon count = smoother models")
print("   • Models are automatically saved to Google Drive")
print("   • The 3D viewer is interactive - drag to rotate, scroll to zoom")
print("\n🎯 Enter your prompt and click GENERATE to create a professional 3D model!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📦 Installing required packages...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 1.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 92.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 92.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
✅ All packages installed!

╔══════════════════════════════════════════════════════════════════╗
║                                


✨ NotY3dGenAI is READY for professional 3D generation!

💡 Tips for best results:
   • Be very descriptive in your prompts
   • Use 'Ultra' or 'High' quality for detailed models
   • Higher polygon count = smoother models
   • Models are automatically saved to Google Drive

🎯 Enter your prompt and click GENERATE to create a professional 3D model!
